In [101]:
library(dplyr)
library(stringr)
library(purrr)
library(progress)

In [102]:
calculate_xG <- function(frame, complement_xg, team, offense, n_next = 10) {
  n <- length(frame)
  out <- rep(NA_real_, n)

  for (i in seq_len(n)) {
    if (i == n) next

    window_idx <- (i + 1):min(i + n_next, n) #gets next 10 rows or til last play
    window_idx <- window_idx[frame[window_idx] != frame[i]] #gets future frames which are not equal to current frame
    
    
    if (length(window_idx) == 0) next

    keep <- if (offense) {
      team[window_idx] == team[i]
    } else {
      team[window_idx] != team[i] 
    }
    vals <- complement_xg[window_idx[keep]]

    out[i] <- if (all(is.na(vals))) NA_real_ else prod(vals, na.rm = TRUE)
  }
  out <- ifelse(is.na(out), 0, 1 - out)
}


In [103]:
dir_path <- "/home/lz80/rdf/sp161/shared/soccer-decision-making-r/sportec/KPI_Merged_all"
csv_files <- list.files(dir_path, pattern = "\\.csv$", full.names = TRUE)

pb <- progress_bar$new(
  format = "[:bar] :current/:total (:percent) eta: :eta elapsed: :elapsed",
  total = length(csv_files),
  clear = FALSE,
  show_after = 0,
  stream = stderr()
)

all_labels <- map_dfr(csv_files, function(f) {
  kpi_df <- read.csv(f, sep = ";") |>
    mutate(
      xG = as.numeric(str_replace(xG, ",", ".")),
      complement_xG = 1 - xG
    ) |>
    arrange(FRAME_NUMBER)
  
  kpi_df <- kpi_df |>
    mutate(
      offense_xG = calculate_xG(FRAME_NUMBER, complement_xG, CUID1, TRUE),
      defense_xG = calculate_xG(FRAME_NUMBER, complement_xG, CUID1, FALSE),
      complete = EVALUATION %in% c("successfullyComplete", "successful")
    ) |>
    filter(SUBTYPE == "Pass") |>
    select(EVENT_ID, offense_xG, defense_xG, complete, CUID1, FRAME_NUMBER) |>
    mutate(source_file = basename(f))
  pb$tick()
  kpi_df
})